# Challenge 4 — GAIL on ALE/MsPacman-v5
## Group 7 | Machine Learning — Universidad Distrital

**Algorithm comparison:** DQN (Challenge 1) → PPO (Challenge 3) → **GAIL** (Challenge 4)

### Notebook structure
1. Global configuration
2. Utilities: `make_env`, `AtariActorCritic`, `compute_gae`
3. GAIL Discriminator (`GAILDiscriminator`)
4. Demonstration collection from the best DQN checkpoint
5. Baseline: Behavioral Cloning (BC)
6. GAIL training (PPO as inner RL algorithm)
7. Evaluation and three-way comparison

### Group 7 hypothesis
Ms. Pac-Man has relatively dense rewards, so GAIL may converge **more slowly** than
PPO/DQN because the adversarial reward is a proxy rather than the true game signal.
The key experiment is determining whether **BC alone** is competitive with full GAIL
on this dense-reward game.


## 0. Setup and imports

In [1]:
from __future__ import annotations
import json, os, shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

import ale_py
import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
from stable_baselines3 import DQN

from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

gym.register_envs(ale_py)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')


PyTorch: 2.12.0
CUDA available: False
Device: cpu


## 1. Global configuration

All hyperparameters in one place to simplify ablation studies.

In [2]:
# ── Environment ───────────────────────────────────────────────────────────────
ENV_ID  = 'ALE/MsPacman-v5'
N_STACK = 4
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Best DQN checkpoint (Challenge 1 — exp_02_lr_high, reward ≈ 1747) ─────────
DQN_CHECKPOINT = '../Challenge_1/models/mspacman_dqn'

# ── Demonstrations ────────────────────────────────────────────────────────────
DEMO_STEPS_PRIMARY = 50_000   # primary dataset
DEMO_STEPS_SMALL   =  5_000   # small-dataset ablation
DEMO_PATH_PRIMARY  = 'demos_50k.npz'
DEMO_PATH_SMALL    = 'demos_5k.npz'

# ── Behavioral Cloning ────────────────────────────────────────────────────────
BC_EPOCHS     = 30
BC_BATCH_SIZE = 256
BC_LR         = 1e-4
BC_SAVE_PATH  = 'models/bc_policy.pt'

# ── GAIL ──────────────────────────────────────────────────────────────────────
GAIL_TOTAL_STEPS   = 5_000_000
GAIL_HORIZON       = 1_024
GAIL_N_PPO_EPOCHS  = 4
GAIL_BATCH_SIZE    = 128
GAIL_LR_POLICY     = 2.5e-4
GAIL_LR_DISC       = 3e-4
GAIL_DISC_UPDATES  = 3        # discriminator updates per rollout
GAIL_GAMMA         = 0.99
GAIL_GAE_LAMBDA    = 0.95
GAIL_CLIP_EPS      = 0.2
GAIL_ENT_COEF      = 0.01
GAIL_VF_COEF       = 0.5
GAIL_MAX_GRAD_NORM = 0.5
GAIL_SEED          = 42
GAIL_SAVE_PATH     = 'models/gail_policy.pt'

# ── Evaluation ────────────────────────────────────────────────────────────────
EVAL_EPISODES = 10
EVAL_SEED     = 0

Path('models').mkdir(exist_ok=True)
print('Configuration loaded.')


Configuration loaded.


## 2. Utilities

### `make_env` — preprocessed Atari environment
Preprocessing pipeline identical to Challenges 1 and 3:
- NoopReset, MaxAndSkip (frame_skip=4)
- Grayscale + resize to 84×84
- EpisodicLifeLoss
- FrameStack(4) → observation shape **(4, 84, 84) uint8**

### `AtariActorCritic` — policy + value network
Nature DQN CNN (3 conv layers) → FC 512 → actor and critic heads.

### `compute_gae` — GAE-λ advantage estimation

In [3]:
def make_env(env_id: str = ENV_ID, seed: int = 0,
             render_mode: str | None = None) -> gym.Env:
    """Preprocessed single Atari environment: (4, 84, 84) uint8, channel-first."""
    base = gym.make(env_id, render_mode=render_mode, frameskip=1)
    env  = AtariPreprocessing(base, noop_max=30, frame_skip=4,
                              screen_size=84, grayscale_obs=True,
                              scale_obs=False)
    env  = FrameStackObservation(env, stack_size=4)
    env.reset(seed=seed)
    return env


class AtariActorCritic(nn.Module):
    """Shared CNN with actor and critic heads (Nature DQN architecture).

    Input:  (batch, 4, 84, 84) uint8 or float32 in [0, 255].
    Output: logits (batch, n_actions), value (batch, 1).
    """
    def __init__(self, n_actions: int) -> None:
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
            nn.Flatten(),
        )
        self.fc          = nn.Linear(64 * 7 * 7, 512)
        self.actor_head  = nn.Linear(512, n_actions)
        self.critic_head = nn.Linear(512, 1)

    def forward(self, obs: torch.Tensor):
        x        = obs.float() / 255.0
        features = F.relu(self.fc(self.cnn(x)))
        return self.actor_head(features), self.critic_head(features)


def compute_gae(
    rewards:    list[float],
    values:     list[torch.Tensor],
    dones:      list[bool],
    next_value: float,
    gamma:      float = 0.99,
    lam:        float = 0.95,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Generalised Advantage Estimation (GAE-lambda)."""
    T          = len(rewards)
    advantages = torch.zeros(T)
    last_gae   = 0.0
    for t in reversed(range(T)):
        next_val = next_value if t == T - 1 else values[t + 1].item()
        mask     = 1.0 - float(dones[t])
        delta    = rewards[t] + gamma * next_val * mask - values[t].item()
        last_gae = delta + gamma * lam * mask * last_gae
        advantages[t] = last_gae
    values_t = torch.stack(values).squeeze(-1).detach()
    returns  = advantages + values_t
    return advantages, returns


# Quick sanity check
env_test = make_env(ENV_ID, seed=0)
print('Observation space:', env_test.observation_space)
print('Action space     :', env_test.action_space)
obs_sample, _ = env_test.reset()
obs_arr = np.array(obs_sample)
print('Obs shape (numpy):', obs_arr.shape, '| dtype:', obs_arr.dtype)
env_test.close()

model_test = AtariActorCritic(n_actions=9).to(DEVICE)
dummy = torch.zeros(1, 4, 84, 84).to(DEVICE)
logits_t, val_t = model_test(dummy)
print('Logits shape:', logits_t.shape, '| Value shape:', val_t.shape)


Observation space: Box(0, 255, (4, 84, 84), uint8)
Action space     : Discrete(9)
Obs shape (numpy): (4, 84, 84) | dtype: uint8
Logits shape: torch.Size([1, 9]) | Value shape: torch.Size([1, 1])


A.L.E: Arcade Learning Environment (version 0.10.1+6a7e0ae)
[Powered by Stella]


## 3. GAIL Discriminator

Network $D_\phi(s[, a]) \to P(\text{expert} \mid s, a) \in (0, 1)$.

Used to generate the **adversarial reward**:
$$r_{\text{adv}}(s, a) = \log D_\phi(s, a)$$

The CNN architecture mirrors `AtariActorCritic` for equal representational capacity.
By default, only the state is used (`use_action=False`); the state-action variant
is available for ablation studies.

In [4]:
class GAILDiscriminator(nn.Module):
    """CNN discriminator for GAIL.

    Trained with BCE to distinguish expert transitions from agent transitions.
    Output: P(expert | s[, a]) in (0, 1).
    """

    def __init__(self, n_actions: int, use_action: bool = False) -> None:
        super().__init__()
        self.use_action = use_action
        self.cnn = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
            nn.Flatten(),
        )
        cnn_out = 64 * 7 * 7  # 3136
        fc_in   = cnn_out + n_actions if use_action else cnn_out
        self.fc = nn.Sequential(
            nn.Linear(fc_in, 512), nn.Tanh(),
            nn.Linear(512, 1),     nn.Sigmoid(),
        )

    def forward(self, obs: torch.Tensor,
                actions_onehot: torch.Tensor | None = None) -> torch.Tensor:
        """
        obs: (batch, 4, 84, 84) float32 in [0, 255].
        Returns: (batch,) with P(expert | s[, a]).
        """
        x     = obs.float() / 255.0
        feats = self.cnn(x)
        if self.use_action and actions_onehot is not None:
            feats = torch.cat([feats, actions_onehot], dim=-1)
        return self.fc(feats).squeeze(-1)


print('GAILDiscriminator ready.')


GAILDiscriminator ready.


## 4. Demonstration collection

Loads the best DQN checkpoint from Challenge 1 (`exp_02_lr_high`, reward ≈ 1747)
and records $(s_t, a_t)$ pairs into a compressed `.npz` file.

**Saved format:** `observations` → (N, 4, 84, 84) uint8 (channel-first),
`actions` → (N,) int64.

| Dataset | Steps | Purpose |
|---|---|---|
| `demos_50k.npz` | 50 000 | Primary training dataset |
| `demos_5k.npz`  |  5 000 | Small-dataset ablation |

In [5]:
def collect_demonstrations(
    model_path: str = DQN_CHECKPOINT,
    n_steps:    int = DEMO_STEPS_PRIMARY,
    seed:       int = 42,
    output:     str = DEMO_PATH_PRIMARY,
) -> dict:
    """Roll out the DQN policy and save (obs, action) pairs.

    SB3 VecFrameStack observations are (84, 84, 4) channel-last.
    Each frame is transposed to (4, 84, 84) channel-first to match make_env().
    """
    env   = make_atari_env(ENV_ID, n_envs=1, seed=seed)
    env   = VecFrameStack(env, n_stack=N_STACK)
    model = DQN.load(model_path, env=env)
    print(f'Checkpoint loaded: {model_path}.zip')

    obs_buf: list = []
    act_buf: list = []
    ep_rewards: list[float] = []
    ep_reward = 0.0
    obs = env.reset()

    for step in range(n_steps):
        action, _ = model.predict(obs, deterministic=True)
        # obs[0]: (84, 84, 4) uint8 — channel-last from SB3 VecFrameStack
        # Transpose to (4, 84, 84) channel-first
        obs_buf.append(obs[0].transpose(2, 0, 1))
        act_buf.append(int(action[0]))

        obs, rewards, dones, infos = env.step(action)
        ep_reward += float(rewards[0])
        if dones[0] and infos[0].get('lives', 0) == 0:
            ep_rewards.append(ep_reward)
            ep_reward = 0.0

        if (step + 1) % 10_000 == 0:
            print(f'  {step + 1:>6,}/{n_steps:,} steps collected')

    env.close()

    demos = {
        'observations': np.array(obs_buf, dtype=np.uint8),
        'actions':      np.array(act_buf, dtype=np.int64),
    }
    np.savez_compressed(output, **demos)

    mean_r = float(np.mean(ep_rewards)) if ep_rewards else 0.0
    std_r  = float(np.std(ep_rewards))  if ep_rewards else 0.0
    print(f'Saved {n_steps:,} steps -> {output}')
    print(f'Full episodes: {len(ep_rewards)}')
    print(f'Demonstrator  mean={mean_r:.1f}  std={std_r:.1f}')

    # Write metadata file (required deliverable)
    meta = output.replace('.npz', '_info.txt')
    with open(meta, 'w') as f:
        f.write(f'source_checkpoint : {model_path}.zip\n')
        f.write(f'env_id            : {ENV_ID}\n')
        f.write(f'n_steps           : {n_steps}\n')
        f.write(f'seed              : {seed}\n')
        f.write(f'full_episodes     : {len(ep_rewards)}\n')
        f.write(f'mean_return       : {mean_r:.2f}\n')
        f.write(f'std_return        : {std_r:.2f}\n')
    print(f'Metadata written -> {meta}')
    return demos


In [6]:
# ── Run demonstration collection ──────────────────────────────────────────────
# Primary dataset (50k steps)
if not Path(DEMO_PATH_PRIMARY).exists():
    collect_demonstrations(
        model_path=DQN_CHECKPOINT,
        n_steps=DEMO_STEPS_PRIMARY,
        seed=42,
        output=DEMO_PATH_PRIMARY,
    )
else:
    print(f'{DEMO_PATH_PRIMARY} already exists, skipping collection.')

# Ablation: small dataset (5k steps)
if not Path(DEMO_PATH_SMALL).exists():
    collect_demonstrations(
        model_path=DQN_CHECKPOINT,
        n_steps=DEMO_STEPS_SMALL,
        seed=42,
        output=DEMO_PATH_SMALL,
    )
else:
    print(f'{DEMO_PATH_SMALL} already exists, skipping collection.')

# Verify
d = np.load(DEMO_PATH_PRIMARY)
print('Observations:', d['observations'].shape, d['observations'].dtype)
print('Actions     :', d['actions'].shape,      d['actions'].dtype)
print('Unique actions:', np.unique(d['actions']))


Wrapping the env in a VecTransposeImage.
Checkpoint loaded: ../Challenge_1/models/mspacman_dqn.zip
  10,000/50,000 steps collected
  20,000/50,000 steps collected
  30,000/50,000 steps collected
  40,000/50,000 steps collected
  50,000/50,000 steps collected
Saved 50,000 steps -> demos_50k.npz
Full episodes: 296
Demonstrator  mean=78.1  std=16.2
Metadata written -> demos_50k_info.txt
Wrapping the env in a VecTransposeImage.
Checkpoint loaded: ../Challenge_1/models/mspacman_dqn.zip


/Users/dcamachoospi/Library/Caches/pypoetry/virtualenvs/challenge4-group7-xwZz_CA_-py3.12/lib/python3.12/site-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 5.65GB > 4.66GB
  warnings.warn(


Saved 5,000 steps -> demos_5k.npz
Full episodes: 29
Demonstrator  mean=79.4  std=16.2
Metadata written -> demos_5k_info.txt
Observations: (50000, 4, 84, 84) uint8
Actions     : (50000,) int64
Unique actions: [0 1 2 3 4 5 6 7 8]


## 5. Behavioral Cloning (BC) — supervised baseline

BC minimises the negative log-likelihood:
$$\mathcal{L}_{\text{BC}}(\theta) = -\frac{1}{N}\sum_{i=1}^{N} \log \pi_\theta(a_i \mid s_i)$$

No environment interaction takes place during training.
Establishes the **supervised lower bound** for comparison with GAIL and DQN.

In [7]:
def train_bc(
    demos_path:      str   = DEMO_PATH_PRIMARY,
    n_epochs:        int   = BC_EPOCHS,
    batch_size:      int   = BC_BATCH_SIZE,
    lr:              float = BC_LR,
    device:          str   = DEVICE,
    save_path:       str   = BC_SAVE_PATH,
    tensorboard_log: str   = 'logs/bc',
) -> AtariActorCritic:
    """Train a Behavioral Cloning policy with cross-entropy on the demo dataset."""
    data  = np.load(demos_path)
    obs_t = torch.tensor(data['observations'], dtype=torch.float32)
    act_t = torch.tensor(data['actions'],      dtype=torch.long)

    dataset = TensorDataset(obs_t, act_t)
    loader  = DataLoader(dataset, batch_size=batch_size,
                         shuffle=True, num_workers=0)

    env = make_env(ENV_ID)
    n_actions = env.action_space.n
    env.close()

    model     = AtariActorCritic(n_actions).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    writer = SummaryWriter(log_dir=tensorboard_log)

    print(f'BC training | {len(dataset):,} samples | '
          f'{n_epochs} epochs | batch={batch_size} | lr={lr}')

    global_step = 0
    for epoch in range(n_epochs):
        total_loss = 0.0
        correct    = 0
        total      = 0
        for obs_b, act_b in loader:
            obs_b = obs_b.to(device)
            act_b = act_b.to(device)
            logits, _ = model(obs_b)
            loss = criterion(logits, act_b)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct    += (logits.argmax(-1) == act_b).sum().item()
            total      += len(act_b)
            writer.add_scalar('bc/loss', loss.item(), global_step)
            global_step += 1

        avg_loss = total_loss / len(loader)
        accuracy = correct / total
        writer.add_scalar('bc/epoch_loss', avg_loss,  epoch + 1)
        writer.add_scalar('bc/accuracy',   accuracy,  epoch + 1)
        print(f'  epoch {epoch+1:3d}/{n_epochs}  '
              f'loss={avg_loss:.4f}  acc={accuracy:.3f}')

    torch.save(model.state_dict(), save_path)
    writer.close()
    print(f'BC policy saved -> {save_path}')
    return model.cpu()


In [8]:
# ── Train BC ──────────────────────────────────────────────────────────────────
bc_model = train_bc(
    demos_path=DEMO_PATH_PRIMARY,
    n_epochs=BC_EPOCHS,
    batch_size=BC_BATCH_SIZE,
    lr=BC_LR,
    device=DEVICE,
    save_path=BC_SAVE_PATH,
    tensorboard_log='logs/bc',
)


BC training | 50,000 samples | 30 epochs | batch=256 | lr=0.0001
  epoch   1/30  loss=1.9181  acc=0.243
  epoch   2/30  loss=1.7283  acc=0.344
  epoch   3/30  loss=1.5338  acc=0.432
  epoch   4/30  loss=1.3613  acc=0.500
  epoch   5/30  loss=1.2557  acc=0.543
  epoch   6/30  loss=1.1834  acc=0.565
  epoch   7/30  loss=1.1247  acc=0.586
  epoch   8/30  loss=1.0766  acc=0.606
  epoch   9/30  loss=1.0312  acc=0.623
  epoch  10/30  loss=0.9919  acc=0.638
  epoch  11/30  loss=0.9435  acc=0.657
  epoch  12/30  loss=0.9157  acc=0.667
  epoch  13/30  loss=0.8803  acc=0.685
  epoch  14/30  loss=0.8357  acc=0.705
  epoch  15/30  loss=0.8027  acc=0.720
  epoch  16/30  loss=0.7692  acc=0.733
  epoch  17/30  loss=0.7406  acc=0.745
  epoch  18/30  loss=0.7095  acc=0.756
  epoch  19/30  loss=0.6827  acc=0.765
  epoch  20/30  loss=0.6552  acc=0.776
  epoch  21/30  loss=0.6320  acc=0.785
  epoch  22/30  loss=0.6021  acc=0.795
  epoch  23/30  loss=0.5828  acc=0.802
  epoch  24/30  loss=0.5560  acc=0.812

## 6. GAIL Training

### Objective
$$\min_{\pi_\theta} \max_{D_\phi} \; \mathbb{E}_{(s,a)\sim\pi_\theta}[\log D_\phi(s,a)] + \mathbb{E}_{(s,a)\sim\mathcal{D}}[\log(1-D_\phi(s,a))] - \lambda H(\pi_\theta)$$

### Training loop
At each rollout of `horizon` steps:
1. **Discriminator update:** BCE on expert and agent mini-batches.
2. **Adversarial reward:** $r_{\text{adv}} = \log D_\phi(s)$ **(completely replaces the environment reward)**.
3. **PPO update:** GAE-λ advantages computed over $r_{\text{adv}}$.

The true environment reward is used **only for evaluation/logging**, never for training.

In [9]:
def train_gail(
    demos_path:               str   = DEMO_PATH_PRIMARY,
    total_steps:              int   = GAIL_TOTAL_STEPS,
    horizon:                  int   = GAIL_HORIZON,
    n_ppo_epochs:             int   = GAIL_N_PPO_EPOCHS,
    batch_size:               int   = GAIL_BATCH_SIZE,
    lr_policy:                float = GAIL_LR_POLICY,
    lr_disc:                  float = GAIL_LR_DISC,
    disc_updates_per_rollout: int   = GAIL_DISC_UPDATES,
    gamma:                    float = GAIL_GAMMA,
    gae_lambda:               float = GAIL_GAE_LAMBDA,
    clip_eps:                 float = GAIL_CLIP_EPS,
    ent_coef:                 float = GAIL_ENT_COEF,
    vf_coef:                  float = GAIL_VF_COEF,
    max_grad_norm:            float = GAIL_MAX_GRAD_NORM,
    seed:                     int   = GAIL_SEED,
    device:                   str   = DEVICE,
    save_path:                str   = GAIL_SAVE_PATH,
    tensorboard_log:          str   = 'logs/gail',
) -> tuple:
    """Train GAIL on MsPacman-v5 with PPO as the inner RL algorithm.

    Returns: (policy, discriminator, all_true_returns)
    True returns are for analysis only; they are NEVER used during training.
    """
    # ── load demonstrations ────────────────────────────────────────────────
    data     = np.load(demos_path)
    demo_obs = torch.tensor(data['observations'], dtype=torch.uint8)
    demo_act = torch.tensor(data['actions'],      dtype=torch.long)
    n_demos  = len(demo_obs)
    print(f'Demonstrations loaded: {n_demos:,} steps from {demos_path}')

    env       = make_env(ENV_ID, seed=seed)
    n_actions = env.action_space.n

    policy = AtariActorCritic(n_actions).to(device)
    disc   = GAILDiscriminator(n_actions, use_action=False).to(device)

    opt_policy = optim.Adam(policy.parameters(), lr=lr_policy)
    opt_disc   = optim.Adam(disc.parameters(),   lr=lr_disc)
    bce        = nn.BCELoss()

    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    writer = SummaryWriter(log_dir=tensorboard_log)

    obs, _ = env.reset()
    ep_true_return    = 0.0
    all_true_returns: list[float] = []
    rollout_count     = 0

    print(f'GAIL | device={device} | total_steps={total_steps:,} | '
          f'horizon={horizon} | disc_updates={disc_updates_per_rollout}')

    for global_step in range(0, total_steps, horizon):

        # ── rollout collection ─────────────────────────────────────────────
        obs_buf, act_buf, logp_buf = [], [], []
        val_buf, done_buf = [], []

        for _ in range(horizon):
            obs_t = torch.tensor(
                np.array(obs), dtype=torch.float32
            ).unsqueeze(0).to(device)

            with torch.no_grad():
                logits, value = policy(obs_t)
            dist   = Categorical(logits=logits)
            action = dist.sample()

            obs_buf.append(obs_t.squeeze(0))
            act_buf.append(action)
            logp_buf.append(dist.log_prob(action))
            val_buf.append(value.squeeze())

            obs, env_r, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated
            done_buf.append(done)
            ep_true_return += float(env_r)

            if done:
                all_true_returns.append(ep_true_return)
                ep_true_return = 0.0
                obs, _ = env.reset()

        # ── adversarial reward (replaces environment reward) ───────────────
        obs_stack = torch.stack(obs_buf).to(device)  # (T, 4, 84, 84)
        with torch.no_grad():
            d_scores    = disc(obs_stack)
        adv_rewards = torch.log(d_scores + 1e-8).cpu().tolist()

        # ── GAE advantages ─────────────────────────────────────────────────
        obs_next_t = torch.tensor(
            np.array(obs), dtype=torch.float32
        ).unsqueeze(0).to(device)
        with torch.no_grad():
            _, nv = policy(obs_next_t)

        advantages, returns = compute_gae(
            adv_rewards, val_buf, done_buf, nv.item(), gamma, gae_lambda
        )

        # ── discriminator update ───────────────────────────────────────────
        for _ in range(disc_updates_per_rollout):
            idx_e    = torch.randint(0, n_demos,  (batch_size,))
            e_obs    = demo_obs[idx_e].float().to(device)
            idx_a    = torch.randint(0, horizon, (batch_size,))
            a_obs    = obs_stack[idx_a]
            d_expert = disc(e_obs)
            d_agent  = disc(a_obs)
            loss_disc = (
                bce(d_expert, torch.ones_like(d_expert)) +
                bce(d_agent,  torch.zeros_like(d_agent))
            )
            opt_disc.zero_grad()
            loss_disc.backward()
            opt_disc.step()

        with torch.no_grad():
            d_acc = (
                (d_expert > 0.5).float().mean() +
                (d_agent  < 0.5).float().mean()
            ) / 2

        # ── PPO update ─────────────────────────────────────────────────────
        act_t  = torch.stack(act_buf).to(device)
        logp_t = torch.stack(logp_buf).detach().to(device)
        adv_t  = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        adv_t  = adv_t.to(device)
        ret_t  = returns.to(device)
        idx    = torch.randperm(horizon)

        for _ in range(n_ppo_epochs):
            for start in range(0, horizon, batch_size):
                mb = idx[start:start + batch_size]
                lg, vn  = policy(obs_stack[mb])
                dn      = Categorical(logits=lg)
                lp_new  = dn.log_prob(act_t[mb])
                ent     = dn.entropy().mean()
                ratio   = (lp_new - logp_t[mb]).exp()
                s1      = ratio * adv_t[mb]
                s2      = ratio.clamp(1-clip_eps, 1+clip_eps) * adv_t[mb]
                l_pi    = -torch.min(s1, s2).mean()
                l_vf    = ((vn.squeeze() - ret_t[mb]) ** 2).mean()
                loss    = l_pi + vf_coef * l_vf - ent_coef * ent
                opt_policy.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(policy.parameters(), max_grad_norm)
                opt_policy.step()

        # ── logging ────────────────────────────────────────────────────────
        rollout_count += 1
        writer.add_scalar('gail/disc_loss',     loss_disc.item(), global_step)
        writer.add_scalar('gail/disc_accuracy', d_acc.item(),     global_step)
        writer.add_scalar('gail/adv_reward_mean',
                          float(np.mean(adv_rewards)), global_step)
        if all_true_returns:
            mean_t = float(np.mean(all_true_returns[-100:]))
            writer.add_scalar('gail/true_return_100', mean_t, global_step)
            if rollout_count % 10 == 0:
                print(f'step={global_step:>9,}  true_return={mean_t:7.1f}  '
                      f'disc_loss={loss_disc.item():.3f}  '
                      f'disc_acc={d_acc.item():.2f}')

    env.close()
    writer.close()
    torch.save(policy.state_dict(), save_path)
    print(f'GAIL policy saved -> {save_path}')
    return policy, disc, all_true_returns


In [10]:
# ── Train GAIL (primary: 50k demos) ───────────────────────────────────────────
gail_policy, gail_disc, gail_returns = train_gail(
    demos_path=DEMO_PATH_PRIMARY,
    total_steps=GAIL_TOTAL_STEPS,
    seed=GAIL_SEED,
    save_path=GAIL_SAVE_PATH,
    tensorboard_log='logs/gail_s42',
)


Demonstrations loaded: 50,000 steps from demos_50k.npz
GAIL | device=cpu | total_steps=5,000,000 | horizon=1024 | disc_updates=3
step=    9,216  true_return=  377.1  disc_loss=0.668  disc_acc=0.90
step=   19,456  true_return=  215.9  disc_loss=0.294  disc_acc=0.95
step=   29,696  true_return=  166.4  disc_loss=0.252  disc_acc=0.95
step=   39,936  true_return=  141.1  disc_loss=0.184  disc_acc=0.97
step=   50,176  true_return=  126.2  disc_loss=0.133  disc_acc=0.98
step=   60,416  true_return=   73.6  disc_loss=0.125  disc_acc=0.98
step=   70,656  true_return=   70.0  disc_loss=0.149  disc_acc=0.98
step=   80,896  true_return=   70.0  disc_loss=0.083  disc_acc=0.98
step=   91,136  true_return=   70.0  disc_loss=0.067  disc_acc=0.99
step=  101,376  true_return=   70.0  disc_loss=0.052  disc_acc=1.00
step=  111,616  true_return=   70.0  disc_loss=0.033  disc_acc=1.00
step=  121,856  true_return=   70.0  disc_loss=0.066  disc_acc=1.00
step=  132,096  true_return=   70.0  disc_loss=0.061  d

In [11]:
# ── Ablation: small demonstration set (5k demos) ──────────────────────────────
gail_5k_policy, _, gail_5k_returns = train_gail(
    demos_path=DEMO_PATH_SMALL,
    total_steps=GAIL_TOTAL_STEPS,
    seed=GAIL_SEED,
    save_path='models/gail_5k.pt',
    tensorboard_log='logs/gail_5k',
)


Demonstrations loaded: 5,000 steps from demos_5k.npz
GAIL | device=cpu | total_steps=5,000,000 | horizon=1024 | disc_updates=3
step=    9,216  true_return=  338.9  disc_loss=0.982  disc_acc=0.83
step=   19,456  true_return=  240.3  disc_loss=0.354  disc_acc=0.94
step=   29,696  true_return=  181.6  disc_loss=0.228  disc_acc=0.96
step=   39,936  true_return=  152.9  disc_loss=0.226  disc_acc=0.93
step=   50,176  true_return=  136.0  disc_loss=0.133  disc_acc=0.99
step=   60,416  true_return=   86.6  disc_loss=0.148  disc_acc=0.95
step=   70,656  true_return=   70.0  disc_loss=0.109  disc_acc=0.98
step=   80,896  true_return=   70.0  disc_loss=0.078  disc_acc=0.99
step=   91,136  true_return=   70.0  disc_loss=0.091  disc_acc=0.98
step=  101,376  true_return=   70.0  disc_loss=0.064  disc_acc=0.99
step=  111,616  true_return=   70.0  disc_loss=0.068  disc_acc=0.99
step=  121,856  true_return=   70.0  disc_loss=0.026  disc_acc=0.99
step=  132,096  true_return=   70.0  disc_loss=0.070  dis

## 7. Evaluation and three-way comparison

All algorithms are evaluated under identical conditions:
- Same environment and wrappers
- Same seed (`EVAL_SEED = 0`)
- Same number of episodes
- **Deterministic** action selection (argmax / greedy)

In [12]:
def evaluate_actor_critic(
    model_path: str,
    n_episodes: int = EVAL_EPISODES,
    seed:       int = EVAL_SEED,
    device:     str = DEVICE,
    label:      str = '',
) -> dict:
    """Evaluate a BC or GAIL policy (AtariActorCritic)."""
    env = make_env(ENV_ID, seed=seed)
    n_actions = env.action_space.n
    model = AtariActorCritic(n_actions).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    episode_rewards: list[float] = []
    obs, _ = env.reset()
    ep_reward = 0.0

    while len(episode_rewards) < n_episodes:
        obs_t = torch.tensor(
            np.array(obs), dtype=torch.float32
        ).unsqueeze(0).to(device)
        with torch.no_grad():
            logits, _ = model(obs_t)
        obs, r, terminated, truncated, _ = env.step(logits.argmax(-1).item())
        ep_reward += r
        if terminated or truncated:
            episode_rewards.append(ep_reward)
            print(f'  [{label}] ep {len(episode_rewards):2d}/{n_episodes}  '
                  f'reward={ep_reward:.1f}')
            ep_reward = 0.0
            obs, _ = env.reset()

    env.close()
    arr = np.array(episode_rewards)
    stats = {'mean': arr.mean(), 'std': arr.std(),
             'min': arr.min(),   'max': arr.max()}
    print(f'  [{label}] mean={stats["mean"]:.1f}  std={stats["std"]:.1f}\n')
    return stats


def evaluate_dqn(
    model_path: str = DQN_CHECKPOINT,
    n_episodes: int = EVAL_EPISODES,
    seed:       int = EVAL_SEED,
) -> dict:
    """Evaluate the SB3 DQN model from Challenge 1."""
    env   = make_atari_env(ENV_ID, n_envs=1, seed=seed)
    env   = VecFrameStack(env, n_stack=N_STACK)
    model = DQN.load(model_path, env=env)

    episode_rewards: list[float] = []
    obs = env.reset()
    ep_reward = 0.0

    while len(episode_rewards) < n_episodes:
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = env.step(action)
        ep_reward += float(rewards[0])
        if dones[0] and infos[0].get('lives', 0) == 0:
            episode_rewards.append(ep_reward)
            print(f'  [DQN] ep {len(episode_rewards):2d}/{n_episodes}  '
                  f'reward={ep_reward:.1f}')
            ep_reward = 0.0

    env.close()
    arr = np.array(episode_rewards)
    stats = {'mean': arr.mean(), 'std': arr.std(),
             'min': arr.min(),   'max': arr.max()}
    print(f'  [DQN] mean={stats["mean"]:.1f}  std={stats["std"]:.1f}\n')
    return stats


In [13]:
# ── Three-way comparison ──────────────────────────────────────────────────────
print('=' * 55)
print('Evaluating BC ...')
bc_stats = evaluate_actor_critic(BC_SAVE_PATH, label='BC')

print('Evaluating GAIL (50k demos) ...')
gail_stats = evaluate_actor_critic(GAIL_SAVE_PATH, label='GAIL-50k')

print('Evaluating GAIL (5k demos — ablation) ...')
gail_5k_stats = evaluate_actor_critic('models/gail_5k.pt', label='GAIL-5k')

print('Evaluating DQN (Challenge 1) ...')
dqn_stats = evaluate_dqn(DQN_CHECKPOINT)

# Summary table
print('\n' + '=' * 60)
print(f'{"Algorithm":<22} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
print('-' * 60)
for label, stats in [
    ('BC',               bc_stats),
    ('GAIL (50k demos)', gail_stats),
    ('GAIL (5k demos)',  gail_5k_stats),
    ('DQN — Ch.1',       dqn_stats),
]:
    print(f'{label:<22} {stats["mean"]:>8.1f} {stats["std"]:>8.1f} '
          f'{stats["min"]:>8.1f} {stats["max"]:>8.1f}')
print('=' * 60)


Evaluating BC ...
  [BC] ep  1/10  reward=570.0
  [BC] ep  2/10  reward=560.0
  [BC] ep  3/10  reward=490.0
  [BC] ep  4/10  reward=1090.0
  [BC] ep  5/10  reward=1110.0
  [BC] ep  6/10  reward=720.0
  [BC] ep  7/10  reward=900.0
  [BC] ep  8/10  reward=780.0
  [BC] ep  9/10  reward=1410.0
  [BC] ep 10/10  reward=1030.0
  [BC] mean=866.0  std=280.2

Evaluating GAIL (50k demos) ...
  [GAIL-50k] ep  1/10  reward=70.0
  [GAIL-50k] ep  2/10  reward=70.0
  [GAIL-50k] ep  3/10  reward=70.0
  [GAIL-50k] ep  4/10  reward=70.0
  [GAIL-50k] ep  5/10  reward=70.0
  [GAIL-50k] ep  6/10  reward=70.0
  [GAIL-50k] ep  7/10  reward=70.0
  [GAIL-50k] ep  8/10  reward=70.0
  [GAIL-50k] ep  9/10  reward=70.0
  [GAIL-50k] ep 10/10  reward=70.0
  [GAIL-50k] mean=70.0  std=0.0

Evaluating GAIL (5k demos — ablation) ...
  [GAIL-5k] ep  1/10  reward=70.0
  [GAIL-5k] ep  2/10  reward=70.0
  [GAIL-5k] ep  3/10  reward=70.0
  [GAIL-5k] ep  4/10  reward=70.0
  [GAIL-5k] ep  5/10  reward=70.0
  [GAIL-5k] ep  6/10 

/Users/dcamachoospi/Library/Caches/pypoetry/virtualenvs/challenge4-group7-xwZz_CA_-py3.12/lib/python3.12/site-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 5.65GB > 5.42GB
  warnings.warn(


  [DQN] ep  1/10  reward=61.0
  [DQN] ep  2/10  reward=108.0
  [DQN] ep  3/10  reward=88.0
  [DQN] ep  4/10  reward=59.0
  [DQN] ep  5/10  reward=102.0
  [DQN] ep  6/10  reward=93.0
  [DQN] ep  7/10  reward=81.0
  [DQN] ep  8/10  reward=76.0
  [DQN] ep  9/10  reward=99.0
  [DQN] ep 10/10  reward=89.0
  [DQN] mean=85.6  std=15.6


Algorithm                  Mean      Std      Min      Max
------------------------------------------------------------
BC                        866.0    280.2    490.0   1410.0
GAIL (50k demos)           70.0      0.0     70.0     70.0
GAIL (5k demos)            70.0      0.0     70.0     70.0
DQN — Ch.1                 85.6     15.6     59.0    108.0


## 8. TensorBoard

View all training runs together:

```bash
tensorboard --logdir logs --port 6006
```

### Metrics to report (Challenge 4)
| Metric | BC | GAIL | DQN |
|---|---|---|---|
| Learning curve | — | `gail/true_return_100` | `rollout/ep_rew_mean` |
| Discriminator accuracy | — | `gail/disc_accuracy` | — |
| Discriminator loss | — | `gail/disc_loss` | — |
| Supervised loss | `bc/epoch_loss` | — | `train/loss` |

### Analysis questions (required in the paper)
1. Does GAIL outperform DQN/PPO on MsPacman's dense rewards? Why or why not?
2. How sensitive is GAIL to demonstration dataset size (5k vs 50k)?
3. Does the discriminator collapse to $D \approx 0.5$ or remain informative?
4. In what regime is BC alone competitive with full GAIL or PPO?
